In [16]:
# Getting basin outline using Hydroshed basin boundary from running GEE script

import geopandas as gpd
import pandas as pd
import json
from shapely.geometry import shape
import matplotlib.pyplot as plt
import rasterio
from rasterio.plot import show

# Paths
csv_path = 'Chamoli_3_basins.csv'
output_plot = 'basin_outline.png'

# Load basin outline
df = pd.read_csv(csv_path)
df['geometry'] = df['.geo'].apply(lambda x: shape(json.loads(x)))
gdf = gpd.GeoDataFrame(df, geometry='geometry', crs='EPSG:4326')

# Plot basin outline only
fig, ax = plt.subplots(figsize=(8, 8))

gdf.boundary.plot(ax=ax, edgecolor='black', linewidth=2)

west, south, east, north = gdf.total_bounds

# lsdstreamburn.get_dem wants [lat, lon]
lower_left  = [south, west]   # [min_lat, min_lon]
upper_right = [north, east]   # [max_lat, max_lon]

print("lower_left :", lower_left)
print("upper_right:", upper_right)

lower_left : [22.875001151143998, 75.72499809971262]
upper_right: [31.45833369739237, 82.06666469081911]


In [19]:
import lsdstreamburn.lsdstreamburn as sb
my_dem = sb.get_dem(OT_api_key_fname = "../my_OT_api_key.txt", 
                    source = "COP90", 
                    lower_left = lower_left, 
                    upper_right = upper_right,
                    prefix = "Chamoli_pop_area")

print(my_dem)

I am taking your coordinates from the lower left list
I am taking your coordinates from the upper right list
I am reading you OT API key from the file ../my_OT_api_key.txt
Your source is a 90m DEM.
The grid spacing for your DEM will be:90
I am going to download a file from opentopography (I've removed the API key):
https://portal.opentopography.org/API/globaldem?demtype=COP90&south=22.875001151143998&north=31.45833369739237&west=75.72499809971262&east=82.06666469081911&outputFormat=GTiff
This might take a little while, depending on the size of the file. 
The filename will be:
./Chamoli_pop_area_COP90.tif
The path and file without path are:
./  Chamoli_pop_area_COP90.tif
Finished downloading
./Chamoli_pop_area_COP90.tif


In [20]:
with rasterio.open(my_dem) as src:
    fig, ax = plt.subplots(figsize=(10, 10))

    # Plot DEM
    show(src, ax=ax, cmap='terrain')

    # Reproject basin outlines to DEM CRS if needed
    if gdf.crs != src.crs:
        gdf_plot = gdf.to_crs(src.crs)
    else:
        gdf_plot = gdf

    # Plot basin outline on top
    gdf_plot.boundary.plot(ax=ax, edgecolor='black', linewidth=2)

    ax.set_title("Chamoli basin outline over COP30 DEM")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_aspect('equal', adjustable='box')

    plt.tight_layout()
    plt.savefig(output_plot, dpi=300)
    plt.close(fig)

print("Saved combined plot to:", output_plot)

Saved combined plot to: basin_outline.png
